### Installation & Environment Setup

Run the following command to install required dependencies:

`ash
pip install torch transformers datasets scikit-learn onnx onnxruntime numpy
`

> [!WARNING]
> **UNRUN NOTEBOOK; NO RESULTS CLAIMED.**
> This notebook is an unexecuted template designed for cloud GPU environments (Kaggle T4 or Google Colab). It mirrors the training schema, label mapping, and text hygiene of `scripts/train_detector.py` to fine-tune a compact transformer on `data/splits/train.jsonl.gz` and export an INT8 ONNX student model.

# 🚀 Veritas AI — Notebook 3: GPU Fine-Tuning & ONNX INT8 Export

**Architecture Target**: Cloud GPU (Kaggle T4 / Colab) -> Export to CPU Edge Deployment (2-core CPU, ≤150 MB RAM, zero PyTorch at runtime).


In [ ]:
# Optional: Package local splits for upload to Kaggle / Colab dataset
import os
import glob
import zipfile

def package_splits_for_kaggle(output_zip="data_splits_kaggle.zip"):
    """Zips data/splits/*.jsonl.gz for upload as a Kaggle/Colab dataset."""
    split_files = sorted(glob.glob("data/splits/*.jsonl.gz"))
    if not split_files:
        print("No split files found matching data/splits/*.jsonl.gz")
        return
    with zipfile.ZipFile(output_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for sf in split_files:
            arcname = os.path.basename(sf)
            zf.write(sf, arcname=arcname)
            print(f"Zipped {sf} as {arcname}")
    print(f"Packaged {len(split_files)} split files into {output_zip}")

# Run locally before uploading to Kaggle/Colab:
# package_splits_for_kaggle()


## 1. Install Dependencies & Setup Device


In [ ]:
!pip install -q torch transformers datasets accelerate onnx onnxruntime scikit-learn

import os
import gzip
import json
import re
import unicodedata
import numpy as np
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score
import onnx
from onnxruntime.quantization import quantize_dynamic, QuantType

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active device: {device}")


## 2. Text Hygiene & Input Normalization (Hypothesis H9)

Directly copies text hygiene and strip-markdown logic from `scripts/common/textnorm.py` (no imports from `scripts/` required in cloud environments).


In [ ]:
# Text hygiene: NFKC, zero-width stripping, quote normalization, confusable folding
_ZERO_WIDTH = dict.fromkeys(map(ord, "​‌‍⁠﻿­᠎"), None)
_QUOTES = {
    ord("‘"): "'", ord("’"): "'", ord("‚"): "'", ord("‛"): "'",
    ord("“"): '"', ord("”"): '"', ord("„"): '"', ord("«"): '"', ord("»"): '"'
}
_CONFUSABLES = {
    "а": "a", "е": "e", "о": "o", "р": "p", "с": "c", "х": "x", "у": "y",
    "і": "i", "ј": "j", "ѕ": "s", "ԁ": "d", "һ": "h", "н": "h", "к": "k",
    "А": "A", "В": "B", "Е": "E", "К": "K", "М": "M", "Н": "H", "О": "O",
    "Р": "P", "С": "C", "Т": "T", "Х": "X", "І": "I", "Ј": "J", "Ѕ": "S",
    "ο": "o", "ν": "v", "ι": "i", "ρ": "p", "Α": "A", "Β": "B", "Ε": "E",
    "Η": "H", "Ι": "I", "Κ": "K", "Μ": "M", "Ν": "N", "Ο": "O", "Ρ": "P",
    "Τ": "T", "Χ": "X", "Υ": "Y", "Ζ": "Z",
}
_MIXED = re.compile(r"\w*[A-Za-z]\w*[Ͱ-ϿЀ-ӿ]\w*|\w*[Ͱ-ϿЀ-ӿ]\w*[A-Za-z]\w*")

def _fold_word(m):
    return "".join(_CONFUSABLES.get(ch, ch) for ch in m.group(0))

def normalize_text(text):
    t = unicodedata.normalize("NFKC", text)
    t = t.translate(_ZERO_WIDTH).translate(_QUOTES)
    t = _MIXED.sub(_fold_word, t)
    t = t.replace("\r\n", "\n").replace("\r", "\n").replace(" ", " ")
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r" *\n *", "\n", t)
    return t.strip()

_MD_HEADING = re.compile(r"^\s{0,3}#{1,6}\s+", re.M)
_MD_EMPH = re.compile(r"(\*{1,2}|_{1,2})(?=\S)(.+?)(?<=\S)\1")
_MD_BULLET = re.compile(r"^\s{0,3}[-*+]\s+", re.M)
_MD_RULE = re.compile(r"^\s*([-*_])\1{2,}\s*$", re.M)

def strip_markdown(text):
    t = _MD_RULE.sub("", text)
    t = _MD_HEADING.sub("", t)
    t = _MD_EMPH.sub(r"\2", t)
    t = _MD_BULLET.sub("", t)
    return re.sub(r"\n{3,}", "\n\n", t).strip()

def prep(text):
    """Normalize text and strip markdown formatting artifacts."""
    t = strip_markdown(normalize_text(text))
    return " ".join(t.split())


## 3. Dataset Loading & 4-Class Taxonomy Mapping

Mirrors `label_of(r)` from `scripts/train_detector.py`:
- `0`: `human` (label == 'human')
- `1`: `human_ai_refined` (hybrid with ai_share < 0.5)
- `2`: `ai_ai_refined` (attacked AI or hybrid with ai_share >= 0.5)
- `3`: `ai_generated` (raw AI with attack_id == 'none')


In [ ]:
LABEL_NAMES = {
    0: "human",
    1: "human_ai_refined",
    2: "ai_ai_refined",
    3: "ai_generated",
}

def label_of(record):
    """Assigns ground-truth class according to the 4-class taxonomy."""
    if record.get("origin") == "hybrid":
        return 2 if record.get("ai_share", 0.5) >= 0.5 else 1
    if record.get("label") == "human":
        return 0
    return 3 if record.get("attack_id", "none") == "none" else 2

def load_jsonl_gz(filepath):
    records = []
    with gzip.open(filepath, "rt", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

# Paths for local or Kaggle environment:
train_path = "data/splits/train.jsonl.gz"
dev_path = "data/splits/dev.jsonl.gz"

if os.path.exists(train_path):
    train_records = load_jsonl_gz(train_path)
    dev_records = load_jsonl_gz(dev_path) if os.path.exists(dev_path) else []
    print(f"Loaded {len(train_records)} train records, {len(dev_records)} dev records.")
else:
    print(f"Split {train_path} not found. Upload data_splits_kaggle.zip or run scripts/corpus/build_splits.py.")


## 4. Tokenization & Dataset Construction


In [ ]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess_records(records, max_length=512):
    cleaned_texts = [prep(r["text"]) for r in records]
    labels = [label_of(r) for r in records]
    encodings = tokenizer(cleaned_texts, truncation=True, max_length=max_length, padding=False)
    encodings["labels"] = labels
    return Dataset.from_dict(encodings)

# When ready to train:
# train_dataset = preprocess_records(train_records)
# dev_dataset = preprocess_records(dev_records)


## 5. Fine-Tuning Loop


In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    acc = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro")
    return {"accuracy": acc, "macro_f1": macro_f1}

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4,
    id2label=LABEL_NAMES,
    label2id={v: k for k, v in LABEL_NAMES.items()}
).to(device)

training_args = TrainingArguments(
    output_dir="./results_detector",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    fp16=torch.cuda.is_available(),
    logging_steps=50,
)

# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_dataset,
#     eval_dataset=dev_dataset,
#     tokenizer=tokenizer,
#     data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
#     compute_metrics=compute_metrics,
# )
# trainer.train()


## 6. Export to Standalone ONNX & Dynamic INT8 Quantization


In [ ]:
def export_and_quantize_onnx(model, tokenizer, output_dir="./exported_student"):
    os.makedirs(output_dir, exist_ok=True)
    onnx_fp32_path = os.path.join(output_dir, "student_model.onnx")
    onnx_int8_path = os.path.join(output_dir, "student_model_int8.onnx")

    model.eval()
    dummy_text = "The rapid development of artificial intelligence has reshaped higher education."
    dummy_inputs = tokenizer(dummy_text, return_tensors="pt").to(model.device)

    print("Exporting to FP32 ONNX...")
    torch.onnx.export(
        model,
        (dummy_inputs["input_ids"], dummy_inputs["attention_mask"]),
        onnx_fp32_path,
        input_names=["input_ids", "attention_mask"],
        output_names=["logits"],
        dynamic_axes={
            "input_ids": {0: "batch_size", 1: "sequence_length"},
            "attention_mask": {0: "batch_size", 1: "sequence_length"},
            "logits": {0: "batch_size"}
        },
        opset_version=14,
        do_constant_folding=True
    )

    print("Quantizing to Dynamic INT8...")
    quantize_dynamic(
        model_input=onnx_fp32_path,
        model_output=onnx_int8_path,
        weight_type=QuantType.QInt8
    )

    fp32_size = os.path.getsize(onnx_fp32_path) / (1024 * 1024)
    int8_size = os.path.getsize(onnx_int8_path) / (1024 * 1024)
    print(f"FP32 Model Size: {fp32_size:.2f} MB")
    print(f"INT8 Model Size: {int8_size:.2f} MB (Target: <=25 MB)")

# export_and_quantize_onnx(model, tokenizer)
